# 环节 12 · 非 Linux 平台原生机制演示

纯 Python 标准库，零依赖。手搓四件事：

1. 三平台能力对照矩阵（能力不对等是选型的第一约束）；
2. SBPL 风格的策略判定器（macOS Seatbelt 的 `allow`/`deny` + 默认动作）；
3. AppContainer 风格的能力模型（Windows：声明能力 → 能访问什么）；
4. 「能力缺口」打分：给定需求，各平台缺什么。

> 结论先行：Linux 是"内核给积木"，macOS / Windows 是"系统给一个进程沙箱 + 一套策略"。

In [ ]:
# §1 三平台能力对照矩阵（2=完整 / 1=有限 / 0=不支持）
CAPS = ["文件系统隔离", "网络默认关", "资源限制(CPU/内存)", "进程数限制",
        "syscall 过滤", "视图隔离(PID 等)", "独立内核档"]
PLATFORMS = {
    "Linux":        [2, 2, 2, 2, 2, 2, 2],
    "macOS":        [1, 2, 1, 1, 0, 0, 1],
    "Windows 原生": [1, 1, 2, 2, 0, 0, 2],
    "Windows WSL2": [2, 2, 2, 2, 2, 2, 2],
}
LEVEL = {0: "不支持", 1: "有限", 2: "完整"}

print(f"{'能力':<20}" + "".join(f"{p:<14}" for p in PLATFORMS))
print("-" * 78)
for i, cap in enumerate(CAPS):
    row = "".join(f"{LEVEL[PLATFORMS[p][i]]:<14}" for p in PLATFORMS)
    print(f"{cap:<20}{row}")

print()
for p, vals in PLATFORMS.items():
    score = sum(vals) / (2 * len(CAPS))
    print(f"{p:<14} 能力完整度 {score:.0%}")

print()
print("读法：macOS 缺『资源与 syscall 维度』，Windows 缺『namespace』")
print("     → 这正是『Linux 优先、Windows 靠 WSL2』的结构性原因")

In [ ]:
# §2 SBPL 风格判定器（macOS Seatbelt 的语义：默认动作 + 规则匹配）
import fnmatch


def sbpl_decide(profile, action):
    """规则按书写顺序生效，后者覆盖前者；都不命中则用默认动作。"""
    result = profile["default"]
    for effect, pattern in profile["rules"]:
        if fnmatch.fnmatch(action, pattern):
            result = effect
    return result


PROFILE = {
    "default": "deny",
    "rules": [
        ("allow", "file-read*"),
        ("allow", "process-exec"),
        ("allow", "file-write*"),
        ("deny", "file-read* /etc/*"),      # 显式收紧
        ("deny", "network*"),
    ],
}

ACTIONS = [
    "process-exec",
    "file-read* /usr/bin/python",
    "file-read* /etc/passwd",
    "file-write* /tmp/out.txt",
    "network*",
    "sysctl-read",
]
for a in ACTIONS:
    print(f"{sbpl_decide(PROFILE, a):<6} {a}")

print()
print("注意两点（都在本机实测里出现过）：")
print("  1. 若把 file-read* 收紧成目录白名单，进程可能连启动都做不到 → SIGABRT")
print("  2. 规则是『策略匹配』，越严格不等于越慢")

In [ ]:
# §3 AppContainer 风格的能力模型（Windows：显式声明能力 → 决定能访问什么）
CONTAINER_CAPABILITIES = {
    "internetClient":              {"公网出站"},
    "privateNetworkClientServer":  {"内网出站/入站"},
    "documentsLibrary":            {"文档库"},
    "picturesLibrary":             {"图片库"},
}

RESOURCE_NEEDS = {
    "访问公网 API":      "internetClient",
    "访问内网服务":      "privateNetworkClientServer",
    "读用户文档":        "documentsLibrary",
    "读图片":            "picturesLibrary",
}

def can_access(granted, resource):
    need = RESOURCE_NEEDS.get(resource)
    if need is None:
        return None
    return need in granted


GRANTED = {"internetClient"}          # 只声明了公网
for res in RESOURCE_NEEDS:
    ok = can_access(GRANTED, res)
    print(f"{'允许' if ok else '拒绝 (缺能力)'}  {res}")

print()
print("Windows 的关键差异：没有 namespace，靠『声明了什么能力』决定可达面")
print("Codex 用两个沙箱身份（Offline/Online）来表达『有没有网』，就是同一思路")

In [ ]:
# §4 能力缺口打分：给定需求，各平台缺什么
NEEDS = {
    "禁网":            {"Linux": "net ns", "macOS": "SBPL", "Windows 原生": "防火墙/能力"},
    "限内存 512MB":    {"Linux": "cgroup", "macOS": "ulimit(粗)", "Windows 原生": "Job Object"},
    "限进程数":        {"Linux": "pids.max", "macOS": "ulimit -u", "Windows 原生": "Job Object"},
    "只读工作目录":    {"Linux": "mount ns", "macOS": "SBPL 规则", "Windows 原生": "ACL"},
    "syscall 白名单":  {"Linux": "seccomp", "macOS": None, "Windows 原生": None},
}
GAPS = {"macOS": ["syscall 白名单", "限内存 512MB(粒度粗)", "限进程数(粒度粗)"],
        "Windows 原生": ["syscall 白名单", "只读工作目录(较难)"]}

for plat, gaps in GAPS.items():
    print(f"{plat} 的能力缺口:")
    for g in gaps:
        print(f"  ✗ {g}")
    print()

print("工程含义：上层按『能力』抽象（禁网 / 限内存 / 只读目录），")
print("下层各平台自己映射；做不到的必须显式降级，不能静默忽略")

## §5 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | macOS 有 namespace / cgroup 吗？ | 都没有；Seatbelt 是策略式进程沙箱，资源靠 `ulimit` |
| 2 | Windows 靠什么限资源？ | Job Object；隔离靠受限令牌 + ACL + AppContainer |
| 3 | 为什么 Claude Code 不支持原生 Windows？ | Linux 侧依赖 bubblewrap（需 user ns），WSL1/原生都缺 → 只能 WSL2 |
| 4 | TCC 和沙箱什么关系？ | 两层：沙箱管「能不能调 API」，TCC 管「用户授不授权」 |
| 5 | 三平台最大的差异是什么？ | 不是接口不同，而是**能力不对等**（某些维度在某个平台根本做不到） |
| 6 | 为什么 SBPL 白名单难写？ | 实测收紧读后 `/bin/ls` 直接 SIGABRT（白名单内也崩）→ 必须用系统基线 + 数据驱动 |

**相关长文**：[环节12-非Linux平台原生机制详解.md](./环节12-非Linux平台原生机制详解.md)